Every data structure is ultimately built from primitive types — the fixed-size values that map directly onto hardware. Understanding what they are, how much memory they consume, and where their limits lie gives you a clearer mental model of why certain operations are fast, why overflow bugs happen, and how the type systems of Python and Kotlin differ from each other at the machine level.

## What Are Primitive Types?

A **primitive type** is a data type that is built into the language and maps directly to a fixed number of bytes in memory. There is no object wrapping, no extra metadata — just raw bits.

In statically typed languages like Kotlin (and Java/C), primitives include integers of various sizes, floating-point numbers, booleans, and characters. In Python, *everything* is an object — there are no true primitives — but the same conceptual types exist and Python uses them under the hood in C.

Knowing the size of a type tells you three things:
- How many bytes it occupies in memory
- What range of values it can hold
- What happens if you exceed that range (overflow)

![Primitive Type Sizes](https://raw.githubusercontent.com/schemabotview/dsa/main/img/primitive-type-sizes.svg)

## Integer Types

Integers store whole numbers. The difference between integer types is how many bytes they use — and therefore what range of values they can represent.

| Type | Bytes | Range |
|---|---|---|
| `byte` | 1 | −128 to 127 |
| `short` | 2 | −32,768 to 32,767 |
| `int` | 4 | −2,147,483,648 to 2,147,483,647 (~±2.1 billion) |
| `long` | 8 | −9.2 × 10¹⁸ to 9.2 × 10¹⁸ |

The range of an n-bit signed integer is always −2^(n−1) to 2^(n−1) − 1. One bit is used for the sign; the remaining bits encode the magnitude.

**`int` is the default choice** for most whole-number work. Use `long` when values exceed ~2.1 billion (timestamps in milliseconds, file sizes, large counts). `byte` and `short` are mainly useful in memory-constrained environments or when working with binary protocols.

### Python

In [ ]:
import sys

# Python integers have no fixed size — they grow as needed
small  = 42
big    = 2 ** 100          # no overflow — Python handles arbitrarily large ints
huge   = 10 ** 50

print(f"42          → {sys.getsizeof(small)} bytes")
print(f"2^100       → {sys.getsizeof(big)} bytes")
print(f"10^50       → {sys.getsizeof(huge)} bytes")
print(f"\n2^100 = {big}")

> Python integers are arbitrary-precision — they never overflow. The trade-off is that they are heap-allocated objects with overhead, not raw 4-byte values.

### Kotlin

```kotlin
fun main() {
    // Kotlin has fixed-size integer types — same as Java primitives
    val b: Byte  = 127
    val s: Short = 32_767
    val i: Int   = 2_147_483_647       // Int.MAX_VALUE
    val l: Long  = 9_223_372_036_854_775_807L  // Long.MAX_VALUE

    println(Byte.SIZE_BYTES)   // 1
    println(Short.SIZE_BYTES)  // 2
    println(Int.SIZE_BYTES)    // 4
    println(Long.SIZE_BYTES)   // 8

    println(Int.MIN_VALUE)     // -2147483648
    println(Int.MAX_VALUE)     //  2147483647
}
```

## Integer Overflow

In languages with fixed-size integers, adding 1 to the maximum value wraps around to the minimum value. This is called **overflow** and it is a frequent source of bugs — especially in loop counters, array indices, and financial calculations.

Python is immune to this because its integers grow dynamically. Kotlin (and Java) wrap silently — no exception is thrown.

### Python

In [ ]:
# Python: no overflow — integers grow automatically
x = 2_147_483_647   # equivalent to Int.MAX_VALUE in Kotlin
print(x + 1)        # 2147483648 — no wrap, just a bigger int

### Kotlin

```kotlin
fun main() {
    val max = Int.MAX_VALUE       //  2147483647
    println(max + 1)              // -2147483648  ← wraps to MIN_VALUE silently!

    // Safe alternative — use Long when overflow is possible
    val safeMax = Int.MAX_VALUE.toLong()
    println(safeMax + 1)          //  2147483648  ← correct
}
```

## Floating-Point Types

Floating-point types store numbers with a decimal point. They use the **IEEE 754** standard, which encodes a number as three components packed into a fixed number of bits:

![IEEE 754 Float Layout](https://raw.githubusercontent.com/schemabotview/dsa/main/img/ieee754-float.svg)

- **Sign bit** (1 bit) — 0 for positive, 1 for negative
- **Exponent** (8 bits for `float`, 11 bits for `double`) — encodes the scale as a power of 2
- **Mantissa** (23 bits for `float`, 52 bits for `double`) — stores the significant digits

| Type | Bytes | Significant digits | Use when |
|---|---|---|---|
| `float` | 4 | ~7 | Memory-constrained, graphics, ML weights |
| `double` | 8 | ~15–17 | General-purpose decimal arithmetic |

`double` is the default in most languages. `float` is used in performance-sensitive contexts (GPU computation, large numerical arrays) where halving memory usage matters.

## Floating-Point Precision

Because floats encode values in binary fractions, most decimal numbers cannot be represented exactly — only approximated. This causes the well-known precision issue:

### Python

In [ ]:
# Floating-point precision
print(0.1 + 0.2)          # 0.30000000000000004  — not exactly 0.3
print(0.1 + 0.2 == 0.3)   # False

# Safe comparison — check within a tolerance
tolerance = 1e-9
print(abs(0.1 + 0.2 - 0.3) < tolerance)   # True

# For exact decimal arithmetic (e.g. money), use Decimal
from decimal import Decimal
print(Decimal('0.1') + Decimal('0.2'))     # 0.3  — exact

### Kotlin

```kotlin
fun main() {
    println(0.1 + 0.2)              // 0.30000000000000004
    println(0.1 + 0.2 == 0.3)      // false

    // Safe comparison
    val epsilon = 1e-9
    println(Math.abs(0.1 + 0.2 - 0.3) < epsilon)   // true

    // For exact decimal arithmetic, use BigDecimal
    import java.math.BigDecimal
    println(BigDecimal("0.1").add(BigDecimal("0.2")))  // 0.3
}
```

> **Rule of thumb:** Never use `float` or `double` for currency or anything requiring exact decimal results. Use `Decimal` in Python or `BigDecimal` in Kotlin/Java.

## Boolean

A boolean holds one of two values: `true` or `false`. Logically it needs only 1 bit, but in practice most languages store it as 1 full byte — aligning it to a byte boundary makes it faster to access in memory.

Booleans are what comparisons return and what `if` statements consume. Every conditional in your code ultimately reduces to a boolean.

### Python

In [ ]:
# In Python, bool is a subclass of int — True == 1, False == 0
print(type(True))       # <class 'bool'>
print(True + True)      # 2  — bool arithmetic (rarely useful, good to know)
print(int(True))        # 1

# Truthiness — non-boolean values evaluated in boolean context
falsy_values = [0, 0.0, "", [], {}, None, False]
for v in falsy_values:
    print(f"bool({v!r:6}) = {bool(v)}")

### Kotlin

```kotlin
fun main() {
    val flag: Boolean = true
    println(Boolean.SIZE_BYTES)       // 1

    // Kotlin does NOT have truthiness — only actual booleans work in if/while
    val items = listOf(1, 2, 3)
    if (items.isNotEmpty()) {         // must be explicit — if (items) would not compile
        println("has items")
    }
}
```

## Characters and Text Encoding

A `char` stores a single character. But characters are just numbers — every character is mapped to a numeric code point. The most important encoding systems are:

- **ASCII** — 7 bits, 128 characters (English letters, digits, basic punctuation)
- **Unicode** — the universal standard, covering every character in every language (over 140,000 code points)
- **UTF-8** — variable-width encoding of Unicode (1–4 bytes per character). Used in Python strings and most modern text.
- **UTF-16** — fixed 2-byte encoding for most common characters. Used by the JVM (Kotlin/Java `Char`).

In ASCII, the character `'A'` is 65, `'a'` is 97, `'0'` is 48. The difference between uppercase and lowercase letters is always 32.

### Python

In [ ]:
# Python has no separate char type — a single character is just a string of length 1

ch = 'A'
print(ord(ch))           # 65  — Unicode code point
print(chr(65))           # 'A' — code point to character
print(ord('a') - ord('A'))  # 32  — consistent offset for all ASCII letters

# Unicode — Python strings handle all languages natively
emoji = '😊'
print(ord(emoji))        # 128522
print(len(emoji))        # 1  — one character
print(len(emoji.encode('utf-8')))  # 4  — but 4 bytes in UTF-8

### Kotlin

```kotlin
fun main() {
    val ch: Char = 'A'
    println(ch.code)             // 65
    println(Char.SIZE_BYTES)     // 2  — JVM uses UTF-16

    // Char arithmetic is common in DSA problems
    println('a' - 'A')           // 32
    println('Z' - 'A' + 1)       // 26  — number of uppercase letters
    println('A' + 3)             // D

    // Convert digit char to its integer value
    val digitChar = '7'
    println(digitChar - '0')     // 7  — common DSA trick
}
```

## Python's Type System vs Kotlin's

The two languages take fundamentally different approaches to types, and the difference matters for how you think about memory:

| | Python | Kotlin |
|---|---|---|
| Type system | Dynamic — checked at runtime | Static — checked at compile time |
| Primitives | No true primitives — everything is an object on the heap | Real primitives (`Int`, `Long`, `Boolean`, etc.) stored on the stack |
| Integer size | Arbitrary-precision — grows as needed | Fixed-size — `Int` is 4 bytes, `Long` is 8 bytes |
| Overflow | Never — Python handles it automatically | Silent wrap-around — `Int.MAX_VALUE + 1` becomes `Int.MIN_VALUE` |
| Null safety | Any variable can be `None` | Non-nullable by default — nullable types require explicit `?` |
| Boxing | N/A (everything is already boxed) | Auto-boxing when primitives go into collections |

In Kotlin, `Int` compiles to a JVM `int` primitive (4 bytes, stack) when used as a local variable. But the moment you put it in a `List<Int>`, the JVM boxes it into an `Integer` object on the heap. For performance-critical code with large numerical arrays, Kotlin's `IntArray` avoids boxing by using a raw `int[]` under the hood.

### Python

In [ ]:
import sys

# Python: everything is a heap object — even a simple integer
x = 1
print(sys.getsizeof(x))    # 28 bytes — int object with type info, ref count, value

# Dynamic typing — a variable can hold any type
value = 42
print(type(value))         # <class 'int'>
value = "now a string"
print(type(value))         # <class 'str'>

### Kotlin

```kotlin
fun main() {
    // Primitive on the stack — 4 bytes
    val a: Int = 42

    // Boxed Integer on the heap — same value, much more overhead
    val b: Int? = 42          // nullable Int forces boxing

    // IntArray avoids boxing — backed by a raw int[] on the JVM
    val primitiveArray = IntArray(1_000_000)    // ~4 MB
    val boxedList = ArrayList<Int>(1_000_000)   // ~16 MB  (each Int is an object)

    // Static typing — type is fixed at declaration
    var score: Int = 100
    // score = "oops"   // compile error — type mismatch
}
```

## Type Conversion

Converting between types is explicit in both languages — neither will silently coerce a value in a way that could lose information.

### Python

In [ ]:
# int ↔ float ↔ str conversions
print(int(3.9))       # 3    — truncates, does NOT round
print(float(7))       # 7.0
print(str(42))        # '42'
print(int('255'))     # 255
print(int('ff', 16))  # 255  — parse hex string
print(bin(255))       # '0b11111111'
print(hex(255))       # '0xff'

### Kotlin

```kotlin
fun main() {
    // Explicit conversions — no implicit widening like Java
    val i: Int   = 42
    val l: Long  = i.toLong()
    val f: Float = i.toFloat()
    val d: Double = i.toDouble()
    val s: String = i.toString()

    // Narrowing — may lose information
    val big: Long = 9_999_999_999L
    val narrow: Int = big.toInt()   // 1410065407 — wraps!
    println(narrow)

    // String to number
    val parsed: Int = "255".toInt()
    val hex: Int    = "ff".toInt(16)  // 255
}
```

## Key Takeaways

- **Primitive types** are fixed-size values that map directly to bytes in memory. Their size determines their range, their speed, and their memory footprint.
- **Integer types** differ by size: `byte` (1 byte), `short` (2), `int` (4), `long` (8). Use `int` by default; switch to `long` when values can exceed ~2.1 billion.
- **Integer overflow** wraps silently in Kotlin/Java. Python avoids it entirely with arbitrary-precision integers.
- **Floating-point numbers** are approximations — most decimal fractions cannot be represented exactly in binary. Never use `float`/`double` for exact decimal arithmetic like money.
- **Characters** are just numbers. `'A'` is 65 in ASCII. The `char - '0'` and `char - 'a'` tricks are common in DSA problems.
- **Python** is dynamic and heap-allocates everything. **Kotlin** is static and uses true stack-allocated primitives — which matters for performance-critical code with large arrays.
- **Type conversion is always explicit** — both languages require you to be clear about what you want, preventing silent precision loss.